In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from mlxtend.frequent_patterns import apriori , association_rules
warnings.filterwarnings("ignore" , category=DeprecationWarning)

In [30]:
#Carregar o dataset
df = pd.read_csv("basket.csv")

In [31]:
pd.set_option("display.max_columns",None)
df = df.drop(columns=["transacao"])
print(df.head())

   pao  leite  cafe  manteiga  acucar  arroz  feijao  macarrao  cerveja  \
0    1      1     1         1       0      0       0         0        0   
1    1      1     0         1       0      0       0         0        0   
2    1      1     1         0       1      0       0         0        0   
3    1      1     0         0       0      0       0         0        0   
4    1      1     1         1       1      0       0         0        0   

   refrigerante  
0             0  
1             0  
2             0  
3             0  
4             0  


In [32]:
#Aplicar o algoritmo apriori
frequent_itemsets = apriori(
    df,
    min_support=0.2,
    use_colnames=True
)

print("Itemsets/Subconjuntos frequentes:")
print(frequent_itemsets.sort_values(by = "support", ascending=True))

Itemsets/Subconjuntos frequentes:
    support                 itemsets
7      0.20           (refrigerante)
15     0.20  (cerveja, refrigerante)
12     0.20          (acucar, leite)
10     0.20            (acucar, pao)
17     0.20    (acucar, cafe, leite)
16     0.20      (acucar, cafe, pao)
8      0.25             (leite, pao)
6      0.25                (cerveja)
14     0.25          (arroz, feijao)
11     0.25            (cafe, leite)
4      0.25                  (arroz)
5      0.25                 (feijao)
13     0.30           (acucar, cafe)
3      0.30                 (acucar)
9      0.30              (cafe, pao)
1      0.35                  (leite)
2      0.40                   (cafe)
0      0.40                    (pao)


In [33]:
#Gerar as regras de associação
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

print("Regra de associação")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth",None)
print(rules)

Regra de associação
        antecedents      consequents  antecedent support  consequent support  \
0           (leite)            (pao)                0.35                0.40   
1             (pao)          (leite)                0.40                0.35   
2            (cafe)            (pao)                0.40                0.40   
3             (pao)           (cafe)                0.40                0.40   
4          (acucar)            (pao)                0.30                0.40   
5             (pao)         (acucar)                0.40                0.30   
6            (cafe)          (leite)                0.40                0.35   
7           (leite)           (cafe)                0.35                0.40   
8          (acucar)          (leite)                0.30                0.35   
9           (leite)         (acucar)                0.35                0.30   
10         (acucar)           (cafe)                0.30                0.40   
11           (cafe) 

In [34]:
#Filtrar as regras relevantes
rules_filtered = rules[
    (rules['lift'] > 1) &
    (rules['confidence'] >= 0.6)
].sort_values(by='lift', ascending=False)

print("Regras relevantes")
print(rules_filtered[["antecedents", "consequents","support","confidence","lift"]])

Regras relevantes
        antecedents     consequents  support  confidence      lift
14        (cerveja)  (refrigerante)     0.20    0.800000  4.000000
15   (refrigerante)       (cerveja)     0.20    1.000000  4.000000
12          (arroz)        (feijao)     0.25    1.000000  4.000000
13         (feijao)         (arroz)     0.25    1.000000  4.000000
24    (cafe, leite)        (acucar)     0.20    0.800000  2.666667
25         (acucar)   (cafe, leite)     0.20    0.666667  2.666667
23  (acucar, leite)          (cafe)     0.20    1.000000  2.500000
17    (acucar, pao)          (cafe)     0.20    1.000000  2.500000
10         (acucar)          (cafe)     0.30    1.000000  2.500000
11           (cafe)        (acucar)     0.30    0.750000  2.500000
18      (cafe, pao)        (acucar)     0.20    0.666667  2.222222
19         (acucar)     (cafe, pao)     0.20    0.666667  2.222222
22   (cafe, acucar)         (leite)     0.20    0.666667  1.904762
8          (acucar)         (leite)     0.20

In [35]:
#novas colunas
df

,pao,leite,cafe,manteiga,acucar,arroz,feijao,macarrao,cerveja,refrigerante
0,1,1,1,1,0,0,0,0,0,0
1,1,1,0,1,0,0,0,0,0,0
2,1,1,1,0,1,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0
4,1,1,1,1,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
95,0,0,0,0,0,0,0,0,1,1
96,0,0,0,0,0,0,0,0,1,1
97,0,0,0,0,0,0,0,0,1,1
98,0,0,0,0,0,0,0,0,1,1


In [43]:
#Insights para o gerente
print("Insights para o gerente")

for _, row in rules_filtered.iterrows():
  antecedentes = ", ".join(list(row["antecedents"]))
  consequentes = ", ".join(list(row['consequents']))
  print(
      f"clientes que compram [{antecedentes}]"
      f"tendem a comprar [{consequentes}]"
      f"(confiança=row{row['confidence']:.2f}, lift={row["lift"]:.2f})"
  )

Insights para o gerente
clientes que compram [cerveja]tendem a comprar [refrigerante](confiança=row0.80, lift=4.00)
clientes que compram [refrigerante]tendem a comprar [cerveja](confiança=row1.00, lift=4.00)
clientes que compram [arroz]tendem a comprar [feijao](confiança=row1.00, lift=4.00)
clientes que compram [feijao]tendem a comprar [arroz](confiança=row1.00, lift=4.00)
clientes que compram [cafe, leite]tendem a comprar [acucar](confiança=row0.80, lift=2.67)
clientes que compram [acucar]tendem a comprar [cafe, leite](confiança=row0.67, lift=2.67)
clientes que compram [acucar, leite]tendem a comprar [cafe](confiança=row1.00, lift=2.50)
clientes que compram [acucar, pao]tendem a comprar [cafe](confiança=row1.00, lift=2.50)
clientes que compram [acucar]tendem a comprar [cafe](confiança=row1.00, lift=2.50)
clientes que compram [cafe]tendem a comprar [acucar](confiança=row0.75, lift=2.50)
clientes que compram [cafe, pao]tendem a comprar [acucar](confiança=row0.67, lift=2.22)
clientes que